# Actor-Critic

Теорема о градиенте стратегии связывает градиент целевой функции  и градиент самой стратегии:

$$\nabla_\theta J(\theta) = \mathbb{E}_\pi [Q^\pi(s, a) \nabla_\theta \ln \pi_\theta(a \vert s)]$$

Встает вопрос, как оценить $Q^\pi(s, a)$? В чистом policy-based алгоритме REINFORCE используется отдача $G_t$, полученная методом Монте-Карло в качестве несмещенной оценки $Q^\pi(s, a)$. В Actor-Critic же предлагается отдельно обучать нейронную сеть Q-функции — критика.

Актор-критиком часто называют обобщенный фреймворк (подход), нежели какой-то конкретный алгоритм. Как подход актор-критик не указывает, каким конкретно [policy gradient] методом обучается актор и каким [value based] методом обучается критик. Таким образом актор-критик задает целое [семейство](https://proceedings.neurips.cc/paper_files/paper/1999/file/6449f44a102fde848669bdd9eb6b76fa-Paper.pdf) различных алгоритмов. Рекомендую в качестве шпаргалки использовать упомянутый в тетрадке с REINFORCE [пост из блога Lilian Weng](https://lilianweng.github.io/posts/2018-04-08-policy-gradient/), посвященный наиболее популярным алгоритмам семейства актор-критиков

В данной тетрадке познакомимся с наиболее простым вариантом актор-критика, который так и называют Actor-Critic:

In [ ]:
# Cтавим нужные зависимости, если это колаб
try:
    import google.colab
    COLAB = True
except ModuleNotFoundError:
    COLAB = False
    pass

if COLAB:
    !pip -q install "gymnasium[classic-control, atari, accept-rom-license]"
    !pip -q install piglet
    !pip -q install imageio_ffmpeg
    !pip -q install moviepy==1.0.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.5/67.5 kB 3.2 MB/s eta 0:00:00


In [ ]:
from collections import deque

import gymnasium as gym
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.distributions import Categorical

%matplotlib inline

In [ ]:
env = gym.make("CartPole-v1")
env.reset()

print(f'{env.observation_space=}')
print(f'{env.action_space=}')

n_actions = env.action_space.n
state_dim = env.observation_space.shape
print(f'Action_space: {n_actions} | State_space: {env.observation_space.shape}')

env.observation_space=Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
env.action_space=Discrete(2)
Action_space: 2 | State_space: (4,)


(1 балл)

In [ ]:
def to_tensor(x, dtype=np.float32):
    if isinstance(x, torch.Tensor):
        return x
    x = np.asarray(x, dtype=dtype)
    x = torch.from_numpy(x)
    return x

def symlog(x):
    """Compute symlog values for a vector `x`. It's an inverse operation for symexp."""
    return torch.sign(x) * torch.log(torch.abs(x) + 1)

def symexp(x):
    """Compute symexp values for a vector `x`. It's an inverse operation for symlog."""
    return torch.sign(x) * (torch.exp(torch.abs(x)) - 1.0)


class SymExpModule(nn.Module):
    def forward(self, x):
        return symexp(x)

def select_action_eps_greedy(Q, state, epsilon):
    """Выбирает действие epsilon-жадно."""
    if not isinstance(state, torch.Tensor):
        state = torch.tensor(state, dtype=torch.float32)
    Q_s = Q(state).detach().numpy()

    # action =
    ####### Здесь ваш код ########
    if np.random.rand() < epsilon:
        action = np.random.choice(len(Q_s))
    else:
        action = np.argmax(Q_s)
    ##############################

    action = int(action)
    return action

def sample_batch(replay_buffer, n_samples):
    # sample randomly `n_samples` samples from replay buffer
    # and split an array of samples into arrays: states, actions, rewards, next_actions, terminateds
    ####### Здесь ваш код ########
    indices = np.random.choice(len(replay_buffer), n_samples, replace=False)
    batch = [replay_buffer[i] for i in indices]
    states, actions, rewards, next_states, terminateds = zip(*batch)
    ##############################

    return np.array(states), np.array(actions), np.array(rewards), np.array(next_states), np.array(terminateds)

## Shared-body Actor-Critic

Актор и критик могут обучаться в разных режимах — актор только on-policy (шаг обучения на текущей собранной подтраектории), а критик on-policy или off-policy (шаг обучения на текущей подтраектории или на батче из replay buffer). Это с одной стороны привносит гибкость в обучение, с другой — усложняет его.

Если актор и критик оба обучаются on-policy, то имеет смысл объединить их сетки в одну и делать общий шаг обратного распространения ошибки. Однако, если они обучаются в разных режимах (и с разной частотой обновления), то велика вероятность, что их шаги обучения могут начать конфликтовать в случае общего тела — для такого варианта намного предпочтительнее разделить их на разные подсети (либо аккуратно настраивать гиперпарметры, чтобы стабилизировать обучение). В целом, рекомендуется использовать общий энкодер наблюдений, а далее как можно скорее разделять головы.

Сделаем реализацию актор-критика с общим телом и с on-policy вариантом обучения.

In [ ]:
class ActorBatch:
    def __init__(self):
        self.logprobs = []
        self.q_values = []

    def append(self, log_prob, q_value):
        self.logprobs.append(log_prob)
        self.q_values.append(q_value)

    def clear(self):
        self.logprobs.clear()
        self.q_values.clear()

(3 балла)

In [ ]:
class ActorCriticModel(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim):
        super().__init__()

        # Инициализируйте сеть агента с двумя головами: softmax-актора и линейного критика
        # self.net, self.actor_head, self.critic_head =
        ####### Здесь ваш код ########
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            prev_dim = hidden_dim
        self.net = nn.Sequential(*layers)
        self.actor_head = nn.Linear(prev_dim, output_dim)
        self.critic_head = nn.Linear(prev_dim, output_dim)
        ##############################

    def forward(self, state):
        # Вычислите выбранное действие, логарифм вероятности его выбора и соответствующее значение Q-функции
        ####### Здесь ваш код ########
        features = self.net(state)
        actor_logits = self.actor_head(features)
        critic_values = self.critic_head(features)

        dist = Categorical(logits=actor_logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        Q_s_a = critic_values.gather(1, action.unsqueeze(-1)).squeeze(-1)
        ##############################

        return action, log_prob, Q_s_a

    def evaluate(self, state):
        # Вычислите значения Q-функции для данного состояния
        ####### Здесь ваш код ########
        features = self.net(state)
        q_values = self.critic_head(features)
        ##############################
        return q_values

(6 баллов)

In [ ]:
class ActorCriticAgent:
    def __init__(self, state_dim, action_dim, hidden_dims, lr, gamma, critic_rb_size):
        self.lr = lr
        self.gamma = gamma
        # Инициализируйте модель актор-критика и SGD оптимизатор (например, `torch.optim.Adam)`)
        ####### Здесь ваш код ########
        self.actor_critic = ActorCriticModel(state_dim, hidden_dims, action_dim)

        # Separate learning rates for actor and critic components
        actor_params = list(self.actor_critic.net.parameters()) + list(self.actor_critic.actor_head.parameters())
        critic_params = self.actor_critic.critic_head.parameters()

        self.opt = torch.optim.Adam([
            {'params': actor_params, 'lr': lr * 0.5},
            {'params': critic_params, 'lr': lr}
        ])
        ##############################
        self.actor_batch = ActorBatch()
        self.critic_rb = deque(maxlen=critic_rb_size)

    def act(self, state):
        # Произведите выбор действия и сохраните необходимые данные в батч для последующего обучения
        # Не забудьте сделать q_value.detach()
        # self.actor_batch.append(..)
        ####### Здесь ваш код ########
        state_tensor = to_tensor(state).float().unsqueeze(0)
        action, log_prob, Q_s_a = self.actor_critic(state_tensor)
        self.actor_batch.append(log_prob.squeeze(0), Q_s_a.detach().squeeze(0))
        ##############################
        return action.item()

    def append_to_replay_buffer(self, s, a, r, next_s, terminated):
        # Добавьте новый экземпляр данных в память прецедентов.
        ####### Здесь ваш код ########
        self.critic_rb.append((s, a, r, next_s, terminated))
        ##############################

    def evaluate(self, state):
        return self.actor_critic.evaluate(state)

    def update(self, rollout_size, critic_batch_size, critic_updates_per_actor):
        if len(self.actor_batch.q_values) < rollout_size:
            return
        self.opt.zero_grad()
        loss = self.update_critic(critic_batch_size, critic_updates_per_actor) # Вот тут вызывалось без параметров
        loss += self.update_actor()
        loss.backward()
        self.opt.step()
        self.actor_batch.clear()
        self.critic_rb.clear()

    def update_actor(self):
        Q_s_a = to_tensor(self.actor_batch.q_values).detach()
        logprobs = torch.stack(self.actor_batch.logprobs)
        # Реализуйте шаг обновления актора — вычислите ошибку `loss` и произведите шаг обновления градиентным спуском.
        ####### Здесь ваш код ########
        loss = -torch.mean(Q_s_a * logprobs)
        ##############################
        return loss

    def update_critic(self, batch_size, n_updates=1):
        # Реализуйте n_updates шагов обучения критика.
        ####### Здесь ваш код ########
        total_loss = 0.0
        for _ in range(n_updates):
            if len(self.critic_rb) < batch_size:
                continue
            states, actions, rewards, next_states, terminateds = sample_batch(self.critic_rb, batch_size)
            loss = self.compute_td_loss(states, actions, rewards, next_states, terminateds)
            total_loss += loss.item()
        return total_loss / n_updates if n_updates > 0 else 0.0
        ##############################

    def compute_td_loss(
        self, states, actions, rewards, next_states, terminateds, regularizer=0.1
    ):
        # переводим входные данные в тензоры
        s = to_tensor(states).float()                     # shape: [batch_size, state_size]
        a = to_tensor(actions, int).long()                # shape: [batch_size]
        r = to_tensor(rewards).float()                    # shape: [batch_size]
        s_next = to_tensor(next_states).float()           # shape: [batch_size, state_size]
        term = to_tensor(terminateds, bool)               # shape: [batch_size]

        # получаем Q[s, a] для выбранных действий в текущих состояниях (для каждого примера из батча)
        # Q_s_a = ...
        ####### Здесь ваш код ########
        current_q_values = self.actor_critic.evaluate(s)
        Q_s_a = current_q_values.gather(1, a.unsqueeze(1)).squeeze(1)
        ##############################

        # получаем Q[s_next, *] — значения полезности всех действий в следующих состояниях
        # Q_sn = ...,
        # а затем вычисляем V*[s_next] — оптимальные значения полезности следующих состояний
        # V_sn = ...
        ####### Здесь ваш код ########
        with torch.no_grad():
            next_q_values = self.actor_critic.evaluate(s_next)
            max_next_q_values = torch.max(next_q_values, dim=1)[0]
            V_sn = max_next_q_values * (~term)  # Zero out terminal states
        ##############################

        # вычисляем TD target и далее TD error
        # target = ...
        # td_error = ...
        ####### Здесь ваш код ########
        target = r + self.gamma * V_sn
        td_error = target - Q_s_a
        ##############################

        # MSE loss для минимизации
        loss = torch.mean(td_error ** 2)
        # добавляем регуляризацию на значения Q
        loss += regularizer * Q_s_a.mean()
        return loss


In [ ]:
def run_actor_critic(
        env_name="CartPole-v1",
        hidden_dims=(128, 128), lr=5e-4,
        total_max_steps=200_000,
        train_schedule=16, replay_buffer_size=50000, batch_size=64, critic_updates_per_actor=4,
        eval_schedule=1000, smooth_ret_window=10, success_ret=200.
):
    env = gym.make(env_name)
    episode_return_history = deque(maxlen=smooth_ret_window)

    agent = ActorCriticAgent(
        state_dim=env.observation_space.shape[0], action_dim=env.action_space.n, hidden_dims=hidden_dims,
        lr=lr, gamma=.995, critic_rb_size=replay_buffer_size
    )

    s, _ = env.reset()
    done, episode_return = False, 0.
    eval = False

    for global_step in range(1, total_max_steps+1):
        a = agent.act(s)
        s_next, r, terminated, truncated, _ = env.step(a)
        episode_return += r
        done = terminated or truncated

        # train step
        agent.append_to_replay_buffer(s, a, r, s_next, terminated)
        agent.update(train_schedule, batch_size, critic_updates_per_actor)

        # evaluate
        if global_step % eval_schedule == 0:
            eval = True

        s = s_next
        if done:
            if eval:
                episode_return_history.append(episode_return)
                avg_return = np.mean(episode_return_history)
                print(f'{global_step=} | {avg_return=:.3f}')
                if avg_return >= success_ret:
                    print('Решено!')
                    break

            s, _ = env.reset()
            done, episode_return = False, 0.
            eval = False

run_actor_critic(eval_schedule=2000, total_max_steps=100_000)

global_step=2009 | avg_return=18.000
global_step=4000 | avg_return=14.000
global_step=6001 | avg_return=15.667
global_step=8005 | avg_return=22.250
global_step=10005 | avg_return=22.600
global_step=12008 | avg_return=26.333
global_step=14004 | avg_return=26.571
global_step=16010 | avg_return=26.625
global_step=18030 | avg_return=31.111
global_step=20031 | avg_return=31.600
global_step=22031 | avg_return=33.000
global_step=24017 | avg_return=33.800
global_step=26003 | avg_return=33.500
global_step=28002 | avg_return=30.100
global_step=30003 | avg_return=28.600
global_step=32001 | avg_return=25.100
global_step=34006 | avg_return=23.300
global_step=36002 | avg_return=21.700
global_step=38003 | avg_return=16.000
global_step=40007 | avg_return=13.200
global_step=42001 | avg_return=10.800
global_step=44004 | avg_return=9.900
global_step=46002 | avg_return=9.200
global_step=48008 | avg_return=9.300
global_step=50004 | avg_return=9.200
global_step=52006 | avg_return=9.200
global_step=54002 | a

KeyboardInterrupt: 

In [52]:
#ВОТ ТУТ У МЕНЯ МОДЛЬ С ДВУМЯ ТЕЛАМИ КАК ОБСУЖДАЛИ В ТГ. ОНА СХОДИТСЯ





class SeparatedActorCritic(nn.Module):
    def __init__(self, input_dim, hidden_dims, action_dim):
        super().__init__()
        # АКТОР
        actor_layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            actor_layers.append(nn.Linear(prev_dim, hidden_dim))
            actor_layers.append(nn.ReLU())
            prev_dim = hidden_dim
        actor_layers.append(nn.Linear(prev_dim, action_dim))
        self.actor_net = nn.Sequential(*actor_layers)
        # КРИТИК (ОТДЕЛЬНОЕ ТЕЛО)
        critic_layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            critic_layers.append(nn.Linear(prev_dim, hidden_dim))
            critic_layers.append(nn.ReLU())
            prev_dim = hidden_dim
        critic_layers.append(nn.Linear(prev_dim, action_dim))
        self.critic_net = nn.Sequential(*critic_layers)

    def forward(self, state):
        actor_logits = self.actor_net(state)
        dist = Categorical(logits=actor_logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        q_values = self.critic_net(state)
        Q_s_a = q_values.gather(1, action.unsqueeze(-1)).squeeze(-1)
        return action, log_prob, Q_s_a, q_values

    def evaluate(self, state):
        return self.critic_net(state)

class SeparatedActorCriticAgent:
    def __init__(self, state_dim, action_dim, hidden_dims, actor_lr, critic_lr, gamma, rb_size):
        self.gamma = gamma
        self.actor_critic = SeparatedActorCritic(state_dim, hidden_dims, action_dim)

        # Отдельные оптимизаторы для актора и критика
        self.actor_opt = torch.optim.Adam(self.actor_critic.actor_net.parameters(), lr=actor_lr)
        self.critic_opt = torch.optim.Adam(self.actor_critic.critic_net.parameters(), lr=critic_lr)

        self.actor_batch = ActorBatch()
        self.replay_buffer = deque(maxlen=rb_size)

    def act(self, state):
        state_tensor = to_tensor(state).float().unsqueeze(0)
        action, log_prob, Q_s_a, _ = self.actor_critic(state_tensor)
        self.actor_batch.append(log_prob.squeeze(0), Q_s_a.detach().squeeze(0))
        return action.item()

    def append_to_replay_buffer(self, s, a, r, next_s, terminated):
        self.replay_buffer.append((s, a, r, next_s, terminated))

    def update(self, batch_size, critic_updates):
        if len(self.replay_buffer) < batch_size or len(self.actor_batch.logprobs) == 0:
            return

        self.update_critic(batch_size, critic_updates)
        self.update_actor()

        self.actor_batch.clear()

    def update_actor(self):
        if not self.actor_batch.logprobs:
            return

        logprobs = torch.stack(self.actor_batch.logprobs)
        state_values = to_tensor(self.actor_batch.q_values)

        actor_loss = -(logprobs * state_values.detach()).mean()

        self.actor_opt.zero_grad()
        actor_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.actor_critic.actor_net.parameters(), max_norm=1.0)
        self.actor_opt.step()

    def update_critic(self, batch_size, n_updates):
        if len(self.replay_buffer) < batch_size:
            return

        for _ in range(n_updates):
            states, actions, rewards, next_states, terminateds = sample_batch(self.replay_buffer, batch_size)

            s = to_tensor(states).float()
            a = to_tensor(actions).long()
            r = to_tensor(rewards).float()
            s_next = to_tensor(next_states).float()
            term = to_tensor(terminateds).bool()


            current_q_values = self.actor_critic.evaluate(s)
            q_s_a = current_q_values.gather(1, a.unsqueeze(1)).squeeze(1)


            with torch.no_grad():
                next_q_values = self.actor_critic.evaluate(s_next)
                max_next_q_values = torch.max(next_q_values, dim=1)[0]
                target_q_values = r + self.gamma * max_next_q_values * (~term)


            critic_loss = nn.MSELoss()(q_s_a, target_q_values)

            self.critic_opt.zero_grad()
            critic_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.actor_critic.critic_net.parameters(), max_norm=1.0)
            self.critic_opt.step()



In [56]:

# Тут небольшие правки
def run_separated_actor_critic(
        env_name="CartPole-v1",
        hidden_dims=(256, 256),
        actor_lr=1e-4,
        critic_lr=1e-3,
        total_max_steps=150_000,
        update_every=64,
        rb_size=20000,
        batch_size=128,
        critic_updates=4,
        eval_every=2000,
        smooth_window=20,
        success_threshold=200
):
    env = gym.make(env_name)

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    agent = SeparatedActorCriticAgent(
        state_dim=state_dim,
        action_dim=action_dim,
        hidden_dims=hidden_dims,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        gamma=0.99,
        rb_size=rb_size
    )

    s, _ = env.reset()
    episode_return = 0.0
    episode_returns = deque(maxlen=smooth_window)
    best_avg_return = 0.0

    for step in range(1, total_max_steps + 1):
        a = agent.act(s)
        s_next, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated
        episode_return += r

        agent.append_to_replay_buffer(s, a, r, s_next, terminated)
        s = s_next

        if done:
            episode_returns.append(episode_return)
            s, _ = env.reset()
            episode_return = 0.0

        # Обновление модели
        if step % update_every == 0 and len(agent.replay_buffer) >= batch_size:
            agent.update(batch_size, critic_updates)

        # Оценка производительности
        if step % eval_every == 0 and len(episode_returns) > 0:
            avg_return = np.mean(episode_returns)
            print(f"Step {step}/{total_max_steps} | Avg Return: {avg_return:.2f}")

            if avg_return > best_avg_return:
                best_avg_return = avg_return

            if avg_return >= success_threshold:
                print(f"Environment solved in {step} steps with average return {avg_return:.2f}!")
                break

    env.close()
    return best_avg_return

best_return = run_separated_actor_critic(
    actor_lr=1e-4,
    critic_lr=1e-3,
    total_max_steps=150_000,
    batch_size=64,
    eval_every=2000,
    success_threshold=200
)

Step 2000/150000 | Avg Return: 23.00
Step 4000/150000 | Avg Return: 21.65
Step 6000/150000 | Avg Return: 29.65
Step 8000/150000 | Avg Return: 25.90
Step 10000/150000 | Avg Return: 28.45
Step 12000/150000 | Avg Return: 36.50
Step 14000/150000 | Avg Return: 38.45
Step 16000/150000 | Avg Return: 43.35
Step 18000/150000 | Avg Return: 54.25
Step 20000/150000 | Avg Return: 56.25
Step 22000/150000 | Avg Return: 64.55
Step 24000/150000 | Avg Return: 60.30
Step 26000/150000 | Avg Return: 66.65
Step 28000/150000 | Avg Return: 55.65
Step 30000/150000 | Avg Return: 59.30
Step 32000/150000 | Avg Return: 72.80
Step 34000/150000 | Avg Return: 77.05
Step 36000/150000 | Avg Return: 64.45
Step 38000/150000 | Avg Return: 98.90
Step 40000/150000 | Avg Return: 115.20
Step 42000/150000 | Avg Return: 124.90
Step 44000/150000 | Avg Return: 169.40
Step 46000/150000 | Avg Return: 202.60
Environment solved in 46000 steps with average return 202.60!
